In [4]:
import numpy as jnp

In [5]:
def lower_triangular_map(bins):
    """
    Build a mapping from vectorized lower-triangular indices to a full symmetric matrix.

    Given the number of bins (matrix size), this returns a function that reconstructs
    a symmetric matrix from an array storing only the unique lower-triangular entries.

    Parameters
    ----------
    bins : int
        Size of the symmetric matrix (number of rows/columns).

    Returns
    -------
    symmetric_from_tri : function
        A function that takes an array of lower-triangular elements and
        returns a symmetric `(bins, bins, ...)` array.

    Examples
    --------
    >>> sym_from_tri = lower_triangular_map(3)
    >>> arr = jnp.arange(6)  # lower-triangular entries
    >>> sym_from_tri(arr).shape
    (3, 3)
    """
    sym_shape = jnp.array([bins,bins])
    a, b = jnp.unravel_index(jnp.arange(bins**2), sym_shape)
    a, b = jnp.minimum(a, b), jnp.maximum(a, b)
    map_arr = jnp.array((bins - (a+1)/2)*a + b, dtype=int)

    def symmetric_from_tri(arr):
        s = arr.shape
        return arr[map_arr,...].reshape((bins, bins)+s[1:])
    
    return symmetric_from_tri

In [39]:
bins = 4
sym_from_tri = lower_triangular_map(bins)

# 10 = bins*(bins+1)/2 entries, e.g. 0..9
arr = jnp.arange(10)

# 1) Reconstruct the symmetric matrix
sym = sym_from_tri(arr) 

In [40]:
tri_mask_2d = jnp.tril(jnp.ones((bins, bins), dtype=bool))
masked = jnp.where(tri_mask_2d, sym, -jnp.inf)

In [48]:
tri_mask_2d

array([[ True, False, False, False],
       [ True,  True, False, False],
       [ True,  True,  True, False],
       [ True,  True,  True,  True]])

In [53]:
masked

array([[  0., -inf, -inf, -inf],
       [  1.,   4., -inf, -inf],
       [  2.,   5.,   7., -inf],
       [  3.,   6.,   8.,   9.]])

In [42]:
def LSE(x, axis=None):
    m = jnp.max(x, axis=axis, keepdims=True)
    return (m + jnp.log(jnp.sum(jnp.exp(x - m), axis=axis, keepdims=True))).squeeze()

log_R_pixel_correct = LSE(masked)

In [43]:
log_R_pixel_correct

array(9.45862974)

In [ ]:
tri_mask_2d = jnp.tril(jnp.ones((bins, bins), dtype=bool))
extra = (1,) * (merger_rate_density.ndim - 2)
tri_mask = tri_mask_2d.reshape((bins[0], bins[1]) + extra)
merger_rate_density_tri = jnp.where(tri_mask, merger_rate_density, -jnp.inf)

In [49]:
K = 3
arr_K = jnp.arange(10*K).reshape(10, K)   # lower-tri entries for each of K slices
sym_K = sym_from_tri(arr_K)               # shape (4, 4, K)

extra = (1,) * (sym_K.ndim - 2)           # -> (1,)
tri_mask = tri_mask_2d.reshape((bins, bins) + extra)  # (4,4,1) broadcasts over K
masked_K = jnp.where(tri_mask, sym_K, -jnp.inf)       # (4,4,K)

log_R_pixel_K = LSE(masked_K, axis=(0,1))  

In [51]:
sym_K.shape

(4, 4, 3)

In [52]:
masked_K.shape

(4, 4, 3)

In [3]:
from astropy.cosmology import Planck15
import numpy as np
zs_fixed = np.linspace(1e-5, 0.2, 10)
cosm = np.log(Planck15.differential_comoving_volume(zs_fixed).value)

In [4]:
cosm

array([ 2.15964638, 17.55217863, 18.91685338, 19.70610351, 20.25951825,
       20.68355787, 21.02565675, 21.31112707, 21.55508509, 21.76728714])

In [17]:
import wcosmo
from astropy import units
Planck15_LAL = wcosmo.FlatLambdaCDM(H0=67.90, Om0=0.3065, name="Planck15_LAL")
COSMO = Planck15_LAL

In [20]:
fixed_dvc_dz = 4*np.pi*COSMO.differential_comoving_volume(zs_fixed).to(units.Gpc**3 / units.sr).value

In [22]:
fixed_dvc_dz

array([1.08158062e-07, 5.23628849e-01, 2.05006287e+00, 4.51447228e+00,
       7.85280160e+00, 1.20020407e+01, 1.69004705e+01, 2.24878832e+01,
       2.87057750e+01, 3.54975132e+01])

In [23]:
dVs = Planck15.differential_comoving_volume(zs_fixed) * 4 * np.pi * units.sr
fixed_dvc_dz = dVs.to(units.Gpc**3).value

In [24]:
fixed_dvc_dz

array([1.08926204e-07, 5.27259226e-01, 2.06392971e+00, 4.54424647e+00,
       7.90327048e+00, 1.20771626e+01, 1.70034313e+01, 2.26211538e+01,
       2.88711696e+01, 3.56962467e+01])